# 14 — Análise operacional de triagem (trilha PyHard)

Este notebook consolida os resultados de `data/meta_results/triage_curves.csv` em tabelas e figuras que suportam a discussão do objetivo específico de viabilidade em Data-Centric AI. As métricas analisadas são Precision@k, Recall@k e Lift@k para k ∈ {5%, 10%, 20%, 30%}.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
RESULTS = REPO_ROOT / 'data' / 'meta_results' / 'triage_curves.csv'
FIG_DIR = REPO_ROOT / 'docs' / 'figures'
TABLES_EXPORT = REPO_ROOT / 'docs' / 'tabelas_export'

df = pd.read_csv(RESULTS)
print('Shape:', df.shape)
df.head()

## 1. Tabela consolidada em formato longo (modelo × cenário × k)

In [ ]:
scenario_label = {
    ('in_domain', 'diabetes'): 'Diabetes in-domain',
    ('in_domain', 'covertype'): 'Covertype in-domain',
    ('transfer', 'covertype'): 'Diabetes → Covertype (transfer)',
}
df['scenario'] = df.apply(
    lambda r: scenario_label[(r['setting'], r['test_dataset'])], axis=1,
)
df['k_pct'] = (df['k_frac'] * 100).round(0).astype(int)

tbl = df[[
    'tool', 'scenario', 'model', 'k_pct',
    'precision_at_k', 'recall_at_k', 'lift_at_k',
]].copy()
tbl = tbl.round(4)
tbl.to_csv(TABLES_EXPORT / 'meta_triage_long.csv', index=False)
tbl.head(12)

## 2. Tabela wide — Lift@k por modelo e cenário

In [ ]:
pivot_lift = tbl.pivot_table(
    index=['tool', 'scenario', 'model'], columns='k_pct', values='lift_at_k',
).round(3)
pivot_lift = pivot_lift.reindex(
    index=pd.MultiIndex.from_product([
        ['pyhard', 'hcat'],
        ['Diabetes in-domain', 'Covertype in-domain', 'Diabetes → Covertype (transfer)'],
        ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'XGBoost'],
    ], names=['tool', 'scenario', 'model'])
)
pivot_lift.to_csv(TABLES_EXPORT / 'meta_triage_lift_wide.csv')
pivot_lift

## 3. Tabela wide — Precision@k por modelo e cenário

In [ ]:
pivot_prec = tbl.pivot_table(
    index=['tool', 'scenario', 'model'], columns='k_pct', values='precision_at_k',
).round(3)
pivot_prec = pivot_prec.reindex(pivot_lift.index)
pivot_prec.to_csv(TABLES_EXPORT / 'meta_triage_precision_wide.csv')
pivot_prec

## 4. Figura — Lift@k por cenário (4 modelos)

In [ ]:
order_models = ['Gradient Boosting', 'Logistic Regression', 'XGBoost', 'Random Forest']
order_scenarios = ['Diabetes in-domain', 'Covertype in-domain', 'Diabetes → Covertype (transfer)']
order_tools = [('pyhard', 'PyHard'), ('hcat', 'H-CAT')]

fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), sharey=True, sharex=True)
for row, (tool_key, tool_label) in enumerate(order_tools):
    for col, scen in enumerate(order_scenarios):
        ax = axes[row, col]
        sub = tbl[(tbl['tool'] == tool_key) & (tbl['scenario'] == scen)]
        for name in order_models:
            d = sub[sub['model'] == name].sort_values('k_pct')
            ax.plot(d['k_pct'], d['lift_at_k'], marker='o', label=name, linewidth=2)
        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1,
                   label='Acaso (lift = 1,0)' if (row == 0 and col == 0) else None)
        ax.axhline(1/0.375, color='red', linestyle=':', linewidth=1,
                   label='Teto (1/prev. ≈ 2,67)' if (row == 0 and col == 0) else None)
        if row == 0:
            ax.set_title(scen)
        if row == 1:
            ax.set_xlabel('k (% do conjunto inspecionado)')
        if col == 0:
            ax.set_ylabel(f'{tool_label}\nLift @ k')
        ax.set_xticks([5, 10, 20, 30])
        ax.grid(alpha=0.3)

axes[0, 0].set_ylim(0.4, 2.8)
axes[0, -1].legend(loc='lower right', fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_triage_lift.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Figura — Precision@k e Recall@k lado a lado (só Gradient Boosting, melhor modelo)

In [ ]:
gb = tbl[(tbl['model'] == 'Gradient Boosting') & (tbl['tool'] == 'pyhard')].copy()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for ax, col, title, ref in zip(
    axes, ['precision_at_k', 'recall_at_k'],
    ['Precision @ k', 'Recall @ k'],
    [0.375, None],
):
    for scen in order_scenarios:
        d = gb[gb['scenario'] == scen].sort_values('k_pct')
        ax.plot(d['k_pct'], d[col], marker='o', label=scen, linewidth=2)
    if ref is not None:
        ax.axhline(ref, color='gray', linestyle='--', linewidth=1,
                   label=f'Prevalência ({ref})')
    # Linha de referência recall (seleção aleatória)
    if col == 'recall_at_k':
        xs = np.array([5, 10, 20, 30])
        ax.plot(xs, xs / 100, color='gray', linestyle='--', linewidth=1,
                label='Acaso (recall = k)')
    ax.set_xticks([5, 10, 20, 30])
    ax.set_xlabel('k (% do conjunto inspecionado)')
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Valor')
axes[0].set_ylim(0.0, 1.0)
axes[1].set_ylim(0.0, 1.0)
axes[0].legend(fontsize=8, loc='lower left')
axes[1].legend(fontsize=8, loc='lower right')
plt.suptitle('Triagem com Gradient Boosting (trilha PyHard)', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_triage_precision_recall.png', dpi=160,
            bbox_inches='tight')
plt.show()

## 6. Comparação direta: classificador meta vs. escore composto de medidas isoladas

Reaproveita o lift@k para o escore composto (média das medidas PyHard normalizadas). Serve de baseline para mostrar quanto o classificador meta agrega sobre uma combinação não supervisionada das mesmas medidas.

In [ ]:
baseline = pd.read_csv(
    REPO_ROOT / 'data' / 'comparison_results' / 'comparison_ranking_correlation.csv'
)
baseline['k_pct'] = (baseline['topk'] * 100).round(0).astype(int)
baseline = baseline[['dataset', 'k_pct', 'lift']].rename(columns={'lift': 'lift_baseline'})
baseline['model'] = 'Escore composto (média normalizada)'
baseline['scenario'] = baseline['dataset'].map({
    'diabetes': 'Diabetes in-domain',
    'covertype': 'Covertype in-domain',
})
baseline_wide = baseline.pivot_table(
    index=['scenario', 'model'], columns='k_pct', values='lift_baseline',
).round(3)
baseline_wide

In [ ]:
gb_lift = pivot_lift.loc[
    (['pyhard'], order_scenarios, ['Gradient Boosting']), :
]
gb_lift.index = gb_lift.index.droplevel('tool')
compare = pd.concat([
    baseline_wide,
    gb_lift,
]).sort_index()
compare.to_csv(TABLES_EXPORT / 'meta_triage_vs_composite.csv')
compare

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
k_vals = [5, 10, 20, 30]
for dataset, scen in [('diabetes', 'Diabetes in-domain'), ('covertype', 'Covertype in-domain')]:
    b = baseline[baseline['dataset'] == dataset].sort_values('k_pct')
    gb_row = tbl[(tbl['model'] == 'Gradient Boosting') & (tbl['scenario'] == scen)].sort_values('k_pct')
    ax.plot(b['k_pct'], b['lift_baseline'], marker='s', linestyle='--',
            linewidth=2, label=f'{scen} · escore composto')
    ax.plot(gb_row['k_pct'], gb_row['lift_at_k'], marker='o', linewidth=2,
            label=f'{scen} · classificador meta (GB)')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='Acaso')
ax.set_xlabel('k (% do conjunto inspecionado)')
ax.set_ylabel('Lift @ k')
ax.set_title('Classificador meta vs. escore composto das medidas PyHard')
ax.set_xticks(k_vals)
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_triage_vs_composite.png', dpi=160)
plt.show()

## 7. Regenerar `tabela_top20_resultados_classificadores.md` com os números de triagem apêndice

Para evitar mexer no arquivo principal, exporta um Markdown novo dedicado à triagem.

In [ ]:
lines = []
lines.append('# Análise operacional de triagem — classificador meta PyHard')
lines.append('')
lines.append('Fonte: `data/meta_results/triage_curves.csv`. Scores in-domain obtidos via `cross_val_predict`')
lines.append('(5-fold estratificado, mesma semente dos experimentos principais). Scores de transferência')
lines.append('obtidos por treino em Diabetes e previsão em Covertype, sem reajuste.')
lines.append('')
lines.append('## Lift @ k por modelo e cenário')
lines.append('')
lines.append('| Ferramenta | Cenário | Modelo | k=5% | k=10% | k=20% | k=30% |')
lines.append('|------------|---------|--------|-----:|------:|------:|------:|')
for (tool, scen, name), row in pivot_lift.iterrows():
    vals = [row.get(k, float('nan')) for k in [5, 10, 20, 30]]
    lines.append(f'| {tool.upper()} | {scen} | {name} | {vals[0]:.3f} | {vals[1]:.3f} | {vals[2]:.3f} | {vals[3]:.3f} |')

lines.append('')
lines.append('## Precision @ k por modelo e cenário')
lines.append('')
lines.append('| Ferramenta | Cenário | Modelo | k=5% | k=10% | k=20% | k=30% |')
lines.append('|------------|---------|--------|-----:|------:|------:|------:|')
for (tool, scen, name), row in pivot_prec.iterrows():
    vals = [row.get(k, float('nan')) for k in [5, 10, 20, 30]]
    lines.append(f'| {tool.upper()} | {scen} | {name} | {vals[0]:.3f} | {vals[1]:.3f} | {vals[2]:.3f} | {vals[3]:.3f} |')

lines.append('')
lines.append('## Contraste com escore composto das medidas isoladas (baseline)')
lines.append('')
lines.append('Valores de lift@k do escore composto (média normalizada das medidas PyHard) reaproveitados de `data/comparison_results/comparison_ranking_correlation.csv`.')
lines.append('')
lines.append('| Cenário | Origem | k=5% | k=10% | k=20% | k=30% |')
lines.append('|---------|--------|-----:|------:|------:|------:|')
for _, r in baseline.sort_values(['scenario', 'k_pct']).iterrows():
    # linha só quando k=30 completa, a fim de reaproveitar wide
    pass
for scen in ['Diabetes in-domain', 'Covertype in-domain']:
    b = baseline[baseline['scenario'] == scen].sort_values('k_pct')['lift_baseline'].tolist()
    gb = tbl[(tbl['tool'] == 'pyhard') & (tbl['scenario'] == scen) & (tbl['model'] == 'Gradient Boosting')].sort_values('k_pct')['lift_at_k'].tolist()
    lines.append(f'| {scen} | Escore composto | {b[0]:.3f} | {b[1]:.3f} | {b[2]:.3f} | {b[3]:.3f} |')
    lines.append(f'| {scen} | Classificador meta GB (PyHard) | {gb[0]:.3f} | {gb[1]:.3f} | {gb[2]:.3f} | {gb[3]:.3f} |')

md_content = '\n'.join(lines) + '\n'
(REPO_ROOT / 'docs' / 'tabela_triagem_meta_pyhard.md').write_text(md_content, encoding='utf-8')
print('Arquivo gerado:', REPO_ROOT / 'docs' / 'tabela_triagem_meta_pyhard.md')
print(md_content)